# Understanding the BPE (Byte Pair Encoding) Algorithm

Byte Pair Encoding (BPE) is a data compression technique that merges the most frequent pairs of symbols in a text. In Natural Language Processing, BPE can be used to create subword tokenizers. By repeatedly merging common symbol pairs, BPE constructs a vocabulary of subword units that can effectively handle unknown words and rare tokens.

In this notebook, we walk through a simplified version of the BPE algorithm. We will:

1. Building an initial vocabulary of character-level tokens.
2. Counting adjacent symbol pairs in the vocabulary.
3. Iteratively merging the most frequent pairs to form subword units.
4. Storing the merge operations (ranks) for future encoding.
5. Encoding new words using the learned merges.

In [1]:
from token_utils import Corp, bpe_encode


CORPUS = "The quick brown fox jumps over the dog. The dog jumps over the frog."
VOCAB_SIZE = 15

## 1. Build a Character-Level Vocabulary

We’ll demonstrate BPE on a short sample corpus. The `build_vocabulary` method splits each word into characters and appends an end-of-word symbol (`_` by default).

In [2]:
corp = Corp(end_of_word="_")


# 1. Build the vocabulary
# =======================
corp.build_vocabulary(CORPUS, separate_punctuation=True)


print("Initial Vocabulary (word -> frequency):\n")
for token, freq in corp.vocabulary.items():
    print(f"{token:>20}  {freq}", "__" * freq)

Initial Vocabulary (word -> frequency):

             T h e _  2 ____
         q u i c k _  1 __
         b r o w n _  1 __
             f o x _  1 __
         j u m p s _  2 ____
           o v e r _  2 ____
             t h e _  2 ____
             d o g _  2 ____
                 . _  2 ____
           f r o g _  1 __


**Explanation:**
- Each word from the corpus is converted into space-separated characters plus an underscore.
- Example: `cat` → `c a t _`.
- All occurrences across the corpus are counted, so the dictionary maps `c a t _` → frequency.

## 2. Compute Pair Statistics

We then call `corp.statistics()`, which tallies the frequency of every adjacent pair of symbols.

In [3]:
# 2. Compute the pair statistics
# ==============================
corp.statistics()


corp.statistics()
print("\nPair Statistics:")
for pair, freq in list(corp.pairs.items())[:15]:
    print(f"{pair}  {freq} ", "__" * freq)


Pair Statistics:
('T', 'h')  2  ____
('h', 'e')  4  ________
('e', '_')  4  ________
('q', 'u')  1  __
('u', 'i')  1  __
('i', 'c')  1  __
('c', 'k')  1  __
('k', '_')  1  __
('b', 'r')  1  __
('r', 'o')  2  ____
('o', 'w')  1  __
('w', 'n')  1  __
('n', '_')  1  __
('f', 'o')  1  __
('o', 'x')  1  __


**Explanation:**

- If a word is `c a t _`, its adjacent pairs are `("c", "a")`, `("a", "t")`, `("t", "_")`.
- The pairs dictionary shows how often each pair occurs across all words in the vocabulary.


## 3. Merging the Most Frequent Pairs
When we find the most frequent pair (e.g., `("o", "g")`), we merge it into a single token (`og`). After the merge, we must rebuild the vocabulary to reflect that new merged token.

In [4]:
# 3. Merge the most frequent pair
# ===============================
corp.merge(pair=("o", "g"))  # example pair


print("Vocabulary after merging pair ('o', 'g'):")
for token, freq in corp.merged.items():
    print(f"  {token:>20} {freq}", "__" * freq)  # see: dog and frog

Vocabulary after merging pair ('o', 'g'):
               T h e _ 2 ____
           q u i c k _ 1 __
           b r o w n _ 1 __
               f o x _ 1 __
           j u m p s _ 2 ____
             o v e r _ 2 ____
               t h e _ 2 ____
                d og _ 2 ____
                   . _ 2 ____
              f r og _ 1 __


**Explanation:**

- The merge operation replaces "o g" with "og" in each word where it appears.
- This shortens the tokens in the vocabulary and reduces the total size.


## 4. Learning a Complete BPE Model

Rather than merging once, we typically merge up to a certain number of times (or until no more merges are possible). Each merge is assigned a rank, stored in corp.bpe_ranks. This gives us a final **recipe** of merges.

In [5]:
# 3. Learn the BPE Ranks
# ======================
corp = Corp(end_of_word="_")
corp.build_vocabulary(CORPUS)
corp.fit(num_merges=VOCAB_SIZE)


print("\nLearned BPE Ranks:")
for pair, rank in sorted(corp.bpe_ranks.items(), key=lambda x: x[1]):
    print(f"{str(pair):>20} {rank}")



Learned BPE Ranks:
          ('h', 'e') 0
          ('o', 'g') 1
         ('T', 'he') 2
          ('j', 'u') 3
         ('ju', 'm') 4
        ('jum', 'p') 5
       ('jump', 's') 6
          ('o', 'v') 7
         ('ov', 'e') 8
        ('ove', 'r') 9
         ('t', 'he') 10
         ('d', 'og') 11
          ('q', 'u') 12
         ('qu', 'i') 13
        ('qui', 'c') 14


**Explanation:**

- After each merge, we update the vocabulary and repeat the statistics.
- Pairs discovered earlier get a lower rank (e.g., rank 0, rank 1, …).
- The final bpe_ranks dictionary tells us the order in which merges occurred.


In [6]:
corp.vocabulary

{'The _': 2,
 'quic k _': 1,
 'b r o w n _': 1,
 'f o x _': 1,
 'jumps _': 2,
 'over _': 2,
 'the _': 2,
 'dog . _': 1,
 'dog _': 1,
 'f r og . _': 1}

## 5. Encoding New Words with BPE

With a trained BPE model, we can now encode unfamiliar words by applying the merges in the discovered order.


In [7]:
test_words = ["bulldog", "overload", "frog"]
print("\nTokenization Results:")
for word in test_words:

    # 5. Tokenize the word
    # ====================
    tokens = bpe_encode(word, corp)

    print(f"  {word} -> {tokens}")


Tokenization Results:
  bulldog -> ['b', 'u', 'l', 'l', 'dog', '_']
  overload -> ['over', 'l', 'o', 'a', 'd', '_']
  frog -> ['f', 'r', 'og', '_']


**Explanation:**

- We start by splitting the word into characters plus _.
- We look for adjacent pairs that appear in our bpe_ranks dictionary, merging them from the highest frequency (lowest rank) to the lowest.
- The final list of tokens often preserves meaningful subwords (e.g., `["bull", "dog", "_"]` or `["f", "ro", "g", "_"]`).


## 6. Conclusion
Key Takeaways

1. Building Character-Level Vocabulary: We split each word into individual characters and track frequencies, plus an end-of-word marker.
2. Counting Adjacent Pairs: We identify and count how often pairs of characters (or subwords) appear across the corpus.
3. Merging: We iteratively merge the most frequent pairs, forming subwords that capture meaningful linguistic units.
4. Storing Merge Ranks: Each merged pair is assigned a rank, so we know the exact sequence in which merges happened.
5. Applying Merges: To encode a new word, we repeatedly merge according to our stored ranks, resulting in subword units.
